In [ ]:
# PASO 1 - Importar Perceptron
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from Perceptron_ismael import Perceptron
SEMILLA = 667

In [ ]:
# PREGUNTA 1
# w son los pesos: cuánto influye cada característica en la decisión.
# bias es el umbral desplazado: mueve la frontera para que no tenga que pasar por el origen.
# eta es el learning rate: el tamaño del paso con el que se corrigen w y bias en cada error.
# Una época es una pasada completa por todas las muestras de entrenamiento.
# Si la predicción es correcta, target - pred = 0, así que el incremento es 0 y no se modifica nada.
#
# PREGUNTA 2
# El perceptrón solo puede dibujar una frontera lineal (recta/plano). Si las clases no son
# linealmente separables siempre habrá muestras mal clasificadas, en cada época seguirá
# actualizando los pesos, nunca llega a 0 errores y no converge: los pesos oscilan.

In [ ]:
# PREGUNTA 3 a)
X = [[1, 1], [2, 2], [3, 1], [4, 2]]
y = [-1, -1, +1, +1]
ppn = Perceptron()
ppn.fit(X, y)
print("len(w_):", len(ppn.w_), "- nº características de X:", len(X[0]))
print("len(errores_):", len(ppn.errores_), "- n_iter:", ppn.n_iter)
# PREGUNTA 3 a) len(w_) = 2 = nº de características y len(errores_) = 1000 = n_iter

In [ ]:
# PREGUNTA 3 b)
print("Una muestra:", ppn.predict([1, 1]))
print("Varias muestras:", ppn.predict(X))
# PREGUNTA 3 b) predict() devuelve solo -1 o +1 con una muestra y con varias

In [ ]:
# PREGUNTA 3 c) Tabla OR
X_or = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_or = np.array([-1, 1, 1, 1])
for eta in [0.001, 0.1, 1.0]:
    for semilla in [SEMILLA, 1]:
        ppn_or = Perceptron(eta=eta, n_iter=10000, semilla=semilla).fit(X_or, y_or)
        print(f"eta={eta} semilla={semilla}: errores última época: {ppn_or.errores_[-1]}"
              f", primera época con 0 errores: {ppn_or.errores_.index(0) + 1}"
              f", w_={np.round(ppn_or.w_, 4)}, bias_={ppn_or.bias_:.4f}")
# PREGUNTA 3 c)
# Sí llega a 0 errores: OR es linealmente separable. Con n_iter alto siempre termina con 0 errores
# (en unas 5-12 épocas), así que aumentar n_iter más no cambia nada, solo repite épocas sin errores.
# Cambiar eta escala los pesos y el bias (con eta=1 los pesos son unas 1000 veces mayores que con
# eta=0.001) pero la frontera que separa las clases es equivalente.
# Cambiar la semilla cambia los pesos iniciales, el número de épocas que tarda y los pesos finales,
# pero no cambia el resultado: todas las soluciones clasifican bien las 4 muestras.

In [ ]:
# PREGUNTA 4
# No. Si conocemos la regla exacta y es estable basta con programarla: es exacta, explicable y
# no necesita datos. ML tiene sentido cuando la regla no se conoce, es demasiado compleja o
# cambia con el tiempo, y solo tenemos ejemplos históricos de los que inferir el patrón.

In [ ]:
# PASO 3 - E0: datos originales
df = pd.read_csv("hacer/prestamos_perceptron.csv")
df = df.drop(columns="id")
X = df[["ingresos", "gastos", "importe_prestamo", "interes_pct", "meses_a_devolver"]]
y = df["concedido"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)
ETA = 0.001
N_ITER = 1000
ppn0 = Perceptron(eta=ETA, n_iter=N_ITER, semilla=SEMILLA)
ppn0.fit(X_train, y_train)
acierto0 = accuracy_score(y_test, ppn0.predict(X_test)) * 100
print("Errores última época train:", ppn0.errores_[-1])
print(f"Acierto en test: {acierto0:.2f}%")
plt.plot(range(1, len(ppn0.errores_) + 1), ppn0.errores_)
plt.xlabel("Épocas")
plt.ylabel("Errores")
plt.title("E0: 5 originales sin escalar")
plt.show()
# PREGUNTA 5
# a) No converge: los errores por época oscilan entre 8 y 12 hasta el final sin estabilizarse.
# b) No llega nunca a 0 errores de entrenamiento (mínimo 8, termina con 12 de 28 muestras).
# c) Acierto en test 58.33% (7 de 12) con eta = 0.001 y n_iter = 1000.
# d) Sí. importe_prestamo va de 2500 a 40000 e interes_pct de 3.5 a 9.5. Las actualizaciones
#    w += incremento * x son proporcionales al valor de x, así que las variables grandes (importe, ingresos)
#    dominan los cambios de los pesos y el interés o los meses apenas influyen.

In [ ]:
# PASO 4 - E1: escalado con StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train_escalado = scaler.transform(X_train)
X_test_escalado = scaler.transform(X_test)
ppn1 = Perceptron(eta=ETA, n_iter=N_ITER, semilla=SEMILLA)
ppn1.fit(X_train_escalado, y_train)
acierto1 = accuracy_score(y_test, ppn1.predict(X_test_escalado)) * 100
print("Errores última época train:", ppn1.errores_[-1])
print(f"Acierto en test: {acierto1:.2f}%")
plt.plot(range(1, len(ppn0.errores_) + 1), ppn0.errores_, label="E0 sin escalar")
plt.plot(range(1, len(ppn1.errores_) + 1), ppn1.errores_, label="E1 escalado")
plt.xlabel("Épocas")
plt.ylabel("Errores")
plt.title("E0 vs E1")
plt.legend()
plt.show()
# PREGUNTA 6
# Mejora las dos cosas: los errores de train bajan (de 12 a 4, mínimo 4) y el acierto en test
# sube de 58.33% a 83.33%, aunque sigue sin llegar a 0 errores porque el problema con estas
# 5 variables no es linealmente separable.
# Escalar no cambia qué relaciones son lineales, pero pone todas las variables en la misma
# escala (media 0, desviación 1) y así todas contribuyen por igual a las actualizaciones y el
# bias (que se actualiza con el incremento, sin multiplicar por x) tiene un peso comparable al de los demás.

In [ ]:
# PASO 5 - Características derivadas
# PREGUNTA 7 Experimento A) originales + capital_disponible + cuota_mensual
df["capital_disponible"] = df["ingresos"] - df["gastos"]
df["total_a_devolver"] = df["importe_prestamo"] * (1 + df["interes_pct"] / 100)
df["cuota_mensual"] = df["total_a_devolver"] / df["meses_a_devolver"]
df["ratio_carga"] = df["cuota_mensual"] / df["capital_disponible"]
cols_A = ["ingresos", "gastos", "importe_prestamo", "interes_pct", "meses_a_devolver",
          "capital_disponible", "cuota_mensual"]
X_train_A = df.loc[X_train.index, cols_A]
X_test_A = df.loc[X_test.index, cols_A]
scaler_A = StandardScaler()
scaler_A.fit(X_train_A)
X_train_A = scaler_A.transform(X_train_A)
X_test_A = scaler_A.transform(X_test_A)
ppn2 = Perceptron(eta=ETA, n_iter=N_ITER, semilla=SEMILLA)
ppn2.fit(X_train_A, y_train)
acierto2 = accuracy_score(y_test, ppn2.predict(X_test_A)) * 100
print("Errores última época train:", ppn2.errores_[-1])
print(f"Acierto en test: {acierto2:.2f}%")
plt.plot(range(1, len(ppn2.errores_) + 1), ppn2.errores_)
plt.xlabel("Épocas")
plt.ylabel("Errores")
plt.title("E2: originales + capital + cuota (escalado)")
plt.show()

In [ ]:
# PREGUNTA 7 Experimento B) solo ratio_carga (se eliminan cuota_mensual y capital_disponible)
cols_B = ["ratio_carga"]
X_train_B = df.loc[X_train.index, cols_B]
X_test_B = df.loc[X_test.index, cols_B]
scaler_B = StandardScaler()
scaler_B.fit(X_train_B)
X_train_B = scaler_B.transform(X_train_B)
X_test_B = scaler_B.transform(X_test_B)
ppn3 = Perceptron(eta=ETA, n_iter=N_ITER, semilla=SEMILLA)
ppn3.fit(X_train_B, y_train)
acierto3 = accuracy_score(y_test, ppn3.predict(X_test_B)) * 100
print("Errores última época train:", ppn3.errores_[-1])
print(f"Acierto en test: {acierto3:.2f}%")
plt.plot(range(1, len(ppn3.errores_) + 1), ppn3.errores_)
plt.xlabel("Épocas")
plt.ylabel("Errores")
plt.title("E3: ratio_carga (escalado)")
plt.show()
# PREGUNTA 8
# No ha cambiado el algoritmo: mismo perceptrón, mismo eta, n_iter, semilla y partición.
# Ha cambiado la representación de los datos. Con capital y cuota llega a 0 errores en la época 15
# y 100% en test; con ratio_carga ya en la primera época, porque la regla es ratio_carga <= 1/3,
# una frontera lineal con una sola variable.
# Una buena característica hace que la relación sea lineal y fácil de aprender; ningún algoritmo
# lineal puede aprender bien si la información relevante no está representada de forma útil.

In [ ]:
# PASO 6 - Comparación final de los experimentos
tabla = pd.DataFrame({
    "Experimento": ["E0", "E1", "E2", "E3"],
    "Características": ["5 originales", "5 originales", "originales + capital + cuota", "ratio_carga"],
    "Escalado": ["No", "Sí", "Sí", "Sí"],
    "Errores finales train": [ppn0.errores_[-1], ppn1.errores_[-1], ppn2.errores_[-1], ppn3.errores_[-1]],
    "Acierto test": [acierto0, acierto1, acierto2, acierto3]})
display(tabla)
# Conclusiones:
# E0: no converge, 12 errores en train y 58.33% en test: las escalas tan distintas dificultan aprender.
# E1: escalar mejora (4 errores, 83.33%) pero la regla no es lineal con las 5 originales.
# E2: añadir capital y cuota (escalado) hace el problema separable: 0 errores desde la época 15 y 100%.
# E3: ratio_carga sola reproduce la regla del banco: 0 errores desde la época 1 y 100% en test.